In [6]:
import os
from pydub import AudioSegment,silence
import librosa


In [4]:
import os
from pydub import AudioSegment, silence


def convert_audio_to_mono(audio_path):
    audio = AudioSegment.from_wav(audio_path)
    audio = audio.set_channels(1)
    return audio


def extract_speech_from_audio(audio_dir, output_dir):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    for audio_file in os.listdir(audio_dir):
        audio_path = os.path.join(audio_dir, audio_file)

        # Convert audio to mono (if it's stereo)
        audio = convert_audio_to_mono(audio_path)

        # Split audio into speech segments based on silence
        audio_segments = silence.split_on_silence(
            audio,
            min_silence_len=300,
            silence_thresh=-35,
        )  # Adjust as needed

        # Filter out segments shorter than 6 seconds
        audio_segments = [s for s in audio_segments if 3000 <= len(s) <= 10000]

        # If there are more than 10 segments, take the first 10
        audio_segments = audio_segments[:200]

        # Export speech segments as separate audio files
        for i, speech_segment in enumerate(audio_segments):
            output_path = os.path.join(
                output_dir, f"{os.path.splitext(audio_file)[0]}_speech_{i}.wav"
            )
            speech_segment.export(output_path, format="wav")


# Paths
extracted_audio_dir = "/home/minwell/Documents/python project/voice_clone/data/raw/voice_raw"
speech_extraction_dir = "/home/minwell/Documents/python project/voice_clone/data/processed/"

# Extract speech from the audio files
extract_speech_from_audio(extracted_audio_dir, speech_extraction_dir)
print("Speech extraction completed.")

Speech extraction completed.


In [ ]:
import os
from faster_whisper import WhisperModel

# Cấu hình model
model_size = "large-v3"
model = WhisperModel(model_size, device="cpu", compute_type="int8")

def format_time(seconds):
    """Chuyển thời gian sang format SRT"""
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    ms = int((seconds - int(seconds)) * 1000)
    return f"{h:02}:{m:02}:{s:02},{ms:03}"

def auto_transcribe(folder_path):
    wav_files = sorted([f for f in os.listdir(folder_path) if f.endswith(".wav")])
    metadata = []

    print(f"🚀 Đang bắt đầu Transcribe {len(wav_files)} file...")

    for wav in wav_files:
        path = os.path.join(folder_path, wav)
        segments, info = model.transcribe(path, beam_size=5, language="vi")

        text_full = ""
        srt_lines = []
        index = 1

        for seg in segments:
            text = seg.text.strip()
            text_full += text + " "

            start = format_time(seg.start)
            end = format_time(seg.end)

            srt_lines.append(f"{index}")
            srt_lines.append(f"{start} --> {end}")
            srt_lines.append(text)
            srt_lines.append("")
            index += 1

        # lưu metadata
        metadata.append(f"{wav}|{text_full.strip()}")

        # lưu file subtitle
        srt_path = os.path.join(folder_path, wav.replace(".wav", ".srt"))
        with open(srt_path, "w", encoding="utf-8") as f:
            f.write("\n".join(srt_lines))

        print(f"Done: {wav}")

    # lưu metadata.csv
    with open(os.path.join(folder_path, "metadata.csv"), "w", encoding="utf-8") as f:
        f.write("\n".join(metadata))


# chạy
auto_transcribe("/home/minwell/Documents/python project/voice_clone/data/processed")


🚀 Đang bắt đầu Transcribe 65 file...
Done: vocals1_speech_0.wav
Done: vocals1_speech_1.wav
Done: vocals1_speech_2.wav
Done: vocals1_speech_3.wav
Done: vocals1_speech_4.wav
Done: vocals1_speech_5.wav
Done: vocals2_speech_0.wav
Done: vocals2_speech_1.wav
Done: vocals2_speech_10.wav
Done: vocals2_speech_11.wav
Done: vocals2_speech_12.wav
Done: vocals2_speech_13.wav
Done: vocals2_speech_14.wav
Done: vocals2_speech_15.wav
Done: vocals2_speech_16.wav
Done: vocals2_speech_17.wav
Done: vocals2_speech_18.wav
Done: vocals2_speech_19.wav
Done: vocals2_speech_2.wav
Done: vocals2_speech_20.wav
Done: vocals2_speech_21.wav
Done: vocals2_speech_22.wav
Done: vocals2_speech_23.wav
Done: vocals2_speech_24.wav
Done: vocals2_speech_25.wav
Done: vocals2_speech_26.wav
Done: vocals2_speech_27.wav
Done: vocals2_speech_28.wav
Done: vocals2_speech_29.wav
Done: vocals2_speech_3.wav
Done: vocals2_speech_30.wav
Done: vocals2_speech_31.wav
Done: vocals2_speech_32.wav
Done: vocals2_speech_33.wav
Done: vocals2_speech_